# v3 series — real patient alignment comparison

Compares real-patient performance across the v3 series under varying sim count and adversarial strength.
All runs use lipschitz encoder + WDGRL + MAF5 flow.

| Run | Sims | λ | Epochs | Key question |
|-----|------|---|--------|--------------|
| v3   | 300k | 0.5 | 400 | Baseline: does WDGRL at scale work? |
| v3.1 | 1M   | 0.5 | 600 | More sims — do sharper posteriors hurt real patients? |
| v3.1 1-NN | 1M | 0.5 | 600 | Does routing real patients through nearest unseen sim fix the gap? |
| v3.2 | 1M   | 2.0 | 600 | Stronger adversarial pressure — does tighter alignment (W1 0.57→0.38) recover v3.1 real-patient gap? |

**Key metrics**: Rap (PVR) R²/MAPE, Ras (SVR) R²/MAPE, credible interval calibration at 90% CI.

In [ ]:
import json, sys, h5py
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import pearsonr
from sklearn.neighbors import NearestNeighbors

try:
    ROOT = Path(globals()['_dh'][0]).parent.parent
    assert (ROOT / 'dataset.py').exists()
except:
    ROOT = Path('/home/sa4604/cv-dann-sbi')
sys.path.insert(0, str(ROOT))

from dataset import (
    load_stats, load_manifest, ReducedCVDataset,
    PARAM_KEYS_INFER, N_CHANNELS, T,
)
from models import LipschitzReducedAutoencoderEncoder

WAVE_KEYS_REAL     = ['Prv', 'Pra', 'Pvp', 'Pap']
REAL_DATA          = Path('/home/sa4604/real_data/onebeat_300patients')
SIM_ROOT           = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
STATS_PATH         = ROOT / 'norm_stats.json'
N_POSTERIOR_SAMPLES = 1000
N_PROXY            = 50_000

device = torch.device('cuda:0')
stats    = load_stats(STATS_PATH)
manifest = load_manifest(SIM_ROOT / 'manifest_train.json')
lo_t = torch.tensor([manifest['config']['pvar_low'][k]  for k in PARAM_KEYS_INFER], dtype=torch.float32)
hi_t = torch.tensor([manifest['config']['pvar_high'][k] for k in PARAM_KEYS_INFER], dtype=torch.float32)

rap_idx = PARAM_KEYS_INFER.index('Rap')
ras_idx = PARAM_KEYS_INFER.index('Ras')

print('ROOT:', ROOT)
print('device:', device)

In [ ]:
# Load real patients (shared across all runs)
w = stats['waves']
p = stats['parameters']
wave_mean = torch.tensor([w[k]['mean'] for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
wave_std  = torch.tensor([w[k]['std']  for k in WAVE_KEYS_REAL], dtype=torch.float32).unsqueeze(1)
pas_mean, pas_std = w['Pas']['mean'], w['Pas']['std'] + 1e-8
vlv_std           = w['Vlv']['std'] + 1e-8
hr_mean,  hr_std  = p['HR']['mean'],  p['HR']['std']  + 1e-8

patients = []
for fpath in sorted(REAL_DATA.glob('*.h5')):
    beats_x, gt = [], {}
    with h5py.File(fpath, 'r') as f:
        for bk in sorted(f.keys()):
            if not bk.startswith('beat_'): continue
            g = f[bk]
            waves = np.stack([g[f'waves/{k}'][:].astype(np.float32) for k in WAVE_KEYS_REAL])
            wt    = (torch.from_numpy(waves) - wave_mean) / (wave_std + 1e-8)
            sbp   = float(g['summaries/sbp'][()])
            dbp   = float(g['summaries/dbp'][()])
            map_  = float(g['summaries/map'][()])
            sv    = float(g['summaries/sv'][()])
            hr    = float(g['parameters/HR'][()])
            sc = torch.tensor([
                (map_ - pas_mean) / pas_std,
                (sbp  - pas_mean) / pas_std,
                (dbp  - pas_mean) / pas_std,
                sv    / vlv_std,
                (hr   - hr_mean)  / hr_std,
            ], dtype=torch.float32)
            beats_x.append(torch.cat([wt.reshape(-1), sc]))
            if not gt:
                gt = {k: float(g[f'parameters/{k}'][()]) for k in f[bk]['parameters'].keys()}
                for sk in g['summaries'].keys():
                    gt[sk] = float(g[f'summaries/{sk}'][()])
                cov = g['covariates']
                gt['cohort'] = cov['cohort'][()].decode() if isinstance(cov['cohort'][()], bytes) else str(cov['cohort'][()])
    if beats_x:
        x_beats = torch.stack(beats_x)
        patients.append(dict(file=fpath.stem, x_beats=x_beats, x_avg=x_beats.mean(0), gt=gt))

rap_gt = np.array([pat['gt'].get('PVR', np.nan) for pat in patients])
ras_gt = np.array([pat['gt'].get('SVR', np.nan) for pat in patients])
rap_valid = ~np.isnan(rap_gt) & (rap_gt >= 0)
ras_valid = ~np.isnan(ras_gt) & (ras_gt >= 0)

print(f'Loaded {len(patients)} patients')
print(f'Rap (PVR) valid: {rap_valid.sum()}   Ras (SVR) valid: {ras_valid.sum()}')

In [ ]:
import time

def run_direct_inference(run_name, run_dir, encoder, flow_net):
    """Run direct flow inference on all real patients, cache to npz."""
    cache = run_dir / 'real_posteriors_direct.npz'
    if cache.exists():
        d = np.load(cache)
        print(f'  [{run_name}] loaded from cache: {cache.name}')
        return d['rap_samples'], d['ras_samples']

    print(f'  [{run_name}] running direct inference on {len(patients)} patients...')
    rap_samples_all, ras_samples_all = [], []
    t0 = time.time()
    for pi, pat in enumerate(patients):
        x = pat['x_avg'].unsqueeze(0).to(device)
        with torch.no_grad():
            z       = encoder(x)
            samples = flow_net.sample((N_POSTERIOR_SAMPLES,), condition=z).squeeze(1).cpu()
        rap_samples_all.append(samples[:, rap_idx].numpy())
        ras_samples_all.append(samples[:, ras_idx].numpy())
        if pi % 200 == 0:
            print(f'    {pi}/{len(patients)}  ({time.time()-t0:.0f}s)', end='\r', flush=True)
    rap_s = np.stack(rap_samples_all)
    ras_s = np.stack(ras_samples_all)
    np.savez(cache, rap_samples=rap_s, ras_samples=ras_s)
    print(f'  [{run_name}] done, saved to {cache.name}  ({time.time()-t0:.0f}s)')
    return rap_s, ras_s


def run_proxy_inference(run_name, run_dir, encoder, flow_net, version):
    """1-NN proxy inference using unseen sims as stand-ins for real patients."""
    cache = run_dir / 'real_posteriors_proxy.npz'
    if cache.exists():
        d = np.load(cache)
        print(f'  [{run_name} 1-NN] loaded from cache: {cache.name}')
        return d['rap_samples'], d['ras_samples']

    # Load or build proxy bank
    proxy_cache = run_dir / 'knn_proxy_bank.npz'
    run_info_path = run_dir / f'run_info_v{version}.json'
    with open(run_info_path) as f:
        run_info = json.load(f)
    proxy_start = run_info['data']['n_sims']

    if proxy_cache.exists():
        loaded       = np.load(proxy_cache)
        z_proxy_bank = loaded['z_bank']
        print(f'  [{run_name} 1-NN] proxy bank loaded: {z_proxy_bank.shape}')
    else:
        print(f'  [{run_name} 1-NN] building proxy bank (N={N_PROXY}, start={proxy_start})...')
        proxy_entries = manifest['index'][proxy_start : proxy_start + N_PROXY]
        ds_proxy = ReducedCVDataset(str(SIM_ROOT / 'train'), proxy_entries, stats)
        x_list = [ds_proxy[i][1] for i in range(len(ds_proxy))]
        ds_proxy.close()
        x_proxy = torch.stack(x_list)
        z_parts = []
        with torch.no_grad():
            for i in range(0, len(x_proxy), 256):
                z_parts.append(encoder(x_proxy[i:i+256].to(device)).cpu().numpy())
        z_proxy_bank = np.concatenate(z_parts)
        np.savez(proxy_cache, z_bank=z_proxy_bank)
        print(f'  [{run_name} 1-NN] proxy bank built, saved to {proxy_cache.name}')

    # Encode real patients
    z_real = np.stack([
        encoder(pat['x_avg'].unsqueeze(0).to(device)).squeeze(0).detach().cpu().numpy()
        for pat in patients
    ])

    # 1-NN lookup
    knn = NearestNeighbors(n_neighbors=1, metric='euclidean', n_jobs=-1)
    knn.fit(z_proxy_bank)
    _, nn_idx = knn.kneighbors(z_real)
    nn_z_proxy = z_proxy_bank[nn_idx[:, 0]]  # (N_patients, latent_dim)

    # Flow inference on proxy latents
    print(f'  [{run_name} 1-NN] flow inference on proxy latents...')
    rap_samples_all, ras_samples_all = [], []
    t0 = time.time()
    for pi in range(len(patients)):
        z_pt = torch.from_numpy(nn_z_proxy[pi]).float().unsqueeze(0).to(device)
        with torch.no_grad():
            samples = flow_net.sample((N_POSTERIOR_SAMPLES,), condition=z_pt).squeeze(1).cpu()
        rap_samples_all.append(samples[:, rap_idx].numpy())
        ras_samples_all.append(samples[:, ras_idx].numpy())
    rap_s = np.stack(rap_samples_all)
    ras_s = np.stack(ras_samples_all)
    np.savez(cache, rap_samples=rap_s, ras_samples=ras_s)
    print(f'  [{run_name} 1-NN] done  ({time.time()-t0:.0f}s)')
    return rap_s, ras_s


def coverage_at(samples, gt, valid, alphas):
    cov = np.zeros(len(alphas))
    for ai, alpha in enumerate(alphas):
        lo = np.percentile(samples, (1 - alpha) / 2 * 100, axis=1)
        hi = np.percentile(samples, (100 - (1 - alpha) / 2 * 100), axis=1)
        cov[ai] = ((gt[valid] >= lo[valid]) & (gt[valid] <= hi[valid])).mean()
    return cov


def metrics(pred_samples, gt, valid):
    pred_mean = pred_samples.mean(axis=1)
    p, g = pred_mean[valid], gt[valid]
    mape = np.mean(np.abs(p - g) / (np.abs(g) + 1e-9)) * 100
    r2   = pearsonr(g, p)[0] ** 2
    return mape, r2

In [ ]:
# Run configurations
RUNS = [
    {'label': 'v3 (300k, λ=0.5)',  'name': 'exp-v3_encoder-lipschitz_dann_flow-maf5',   'version': '3',   'proxy': False},
    {'label': 'v3.1 (1M, λ=0.5)',  'name': 'exp-v3.1_encoder-lipschitz_dann_flow-maf5', 'version': '3.1', 'proxy': False},
    {'label': 'v3.1 1-NN proxy',   'name': 'exp-v3.1_encoder-lipschitz_dann_flow-maf5', 'version': '3.1', 'proxy': True},
    {'label': 'v3.2 (1M, λ=2.0)',  'name': 'exp-v3.2_encoder-lipschitz_dann_flow-maf5', 'version': '3.2', 'proxy': False},
]

results = {}  # label -> {'rap_samples', 'ras_samples'}

for run in RUNS:
    run_dir = ROOT / f'outputs/{run["name"]}'
    print(f'\n--- {run["label"]} ---')
    encoder  = LipschitzReducedAutoencoderEncoder(latent_dim=128).to(device)
    encoder.load_state_dict(torch.load(run_dir / 'encoder.pt', map_location=device))
    encoder.eval()
    flow_net = torch.load(run_dir / 'flow_net.pt', map_location=device, weights_only=False)
    flow_net.eval()

    if run['proxy']:
        rap_s, ras_s = run_proxy_inference(run['name'], run_dir, encoder, flow_net, run['version'])
    else:
        rap_s, ras_s = run_direct_inference(run['name'], run_dir, encoder, flow_net)

    results[run['label']] = {'rap': rap_s, 'ras': ras_s}
    del encoder, flow_net  # free VRAM before next load
    torch.cuda.empty_cache()

print('\nAll runs done.')

## Calibration curves — Rap (PVR) and Ras (SVR)

Empirical coverage of nominal credible intervals across the 4 runs.
Ideal: diagonal. Over-confident: below diagonal. Under-confident: above diagonal.

In [ ]:
alphas = np.linspace(0.05, 0.95, 19)

COLORS    = ['steelblue', 'tomato', 'tomato', 'mediumseagreen']
MARKERS   = ['o', 'o', 's', '^']
LINESTYLE = ['-', '-', '--', '-']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, gt, valid, param_name, samples_key in [
    (axes[0], rap_gt, rap_valid, 'Rap (PVR)',  'rap'),
    (axes[1], ras_gt, ras_valid, 'Ras (SVR)',  'ras'),
]:
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='ideal')
    for i, (label, res) in enumerate(results.items()):
        cov = coverage_at(res[samples_key], gt, valid, alphas)
        ax.plot(alphas, cov,
                color=COLORS[i], marker=MARKERS[i], markersize=4,
                linewidth=1.8, linestyle=LINESTYLE[i], alpha=0.85,
                label=label)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Nominal coverage', fontsize=11)
    ax.set_ylabel('Empirical coverage', fontsize=11)
    ax.set_title(f'{param_name} — real patient calibration (n={valid.sum()})', fontsize=11)
    ax.legend(fontsize=9)

fig.suptitle('Real patient credible interval calibration — v3 series', fontsize=13)
plt.tight_layout(); plt.show()

## Summary table — R², MAPE, calibration @ 90% CI

In [ ]:
idx_90 = alphas.searchsorted(0.90)

header = f'{"Run":<22} {"Rap R²":>8} {"Rap MAPE":>10} {"Rap cov@90%":>13} {"Ras R²":>8} {"Ras MAPE":>10} {"Ras cov@90%":>13}'
print(header)
print('-' * len(header))

for label, res in results.items():
    rap_mape, rap_r2 = metrics(res['rap'], rap_gt, rap_valid)
    ras_mape, ras_r2 = metrics(res['ras'], ras_gt, ras_valid)
    rap_cov = coverage_at(res['rap'], rap_gt, rap_valid, alphas)[idx_90]
    ras_cov = coverage_at(res['ras'], ras_gt, ras_valid, alphas)[idx_90]
    print(
        f'{label:<22} {rap_r2:>8.3f} {rap_mape:>9.1f}%  {rap_cov:>12.3f}'
        f'  {ras_r2:>8.3f} {ras_mape:>9.1f}%  {ras_cov:>12.3f}'
    )